# LoRA Serving

The native generation engine supports a single LoRA adapter merged at startup. It does not switch adapters per request. For LoRA principles, see the [Hugging Face documentation](https://huggingface.co/docs/peft/conceptual_guides/lora).

## Serving Single Adaptor（Static LoRA）
| Feature | Description |
|------|------|
| **Fusion Method** | Before inference, the base model and specified LoRA weights are **permanently fused** (irreversible). |
| **Applicable Scenarios** | Scenarios requiring **single LoRA** with pursuit of optimal performance. |
| **Limitations** | After fusion, the base model cannot be restored; the original output before applying LoRA cannot be obtained simultaneously. |

### Usage
- Set `--merge_lora true` (the default) and pass exactly one adapter in `--lora_info`.
- Example: ```--lora_info {"test0":"/mnt/nas1/lora/taoshang_qwen_lora_18000/lora"}```

In [ ]:
def wait_sever_done(server_process, port: int, timeout: int = 1600):
    host = "localhost"
    retry_interval = 1  # Retry interval (seconds)
    start_time = time.time()

    port = str(port)

    logging.info(f"Waiting for pid[{server_process.pid}] to start...\nPort {port}")
    while True:
        try:
            # Try to connect to the specified host and port
            sock = socket.create_connection((host, port), timeout=timeout)
            sock.close()
            logging.info(f"Port {port} started successfully")
            return True
        except (socket.error, ConnectionRefusedError):
            # If connection fails, wait for a while before retrying
            time.sleep(retry_interval)

            if (
                not psutil.pid_exists(server_process.pid)
                or server_process.poll()
            ):
                logging.warning(
                    f"Process:[{server_process.pid}] status abnormal, service startup failed, please check log files"
                )
                return False
            # If waiting time exceeds the preset timeout, give up waiting
            if time.time() - start_time > timeout:
                logging.warning(
                    f"Waiting for port {port} to start timed out, please check log files"
                )
                return False

In [ ]:
port=8090                
server_process = subprocess.Popen(
        ["/opt/conda310/bin/python", "-m", "rtp_llm.start_server",
         "--checkpoint_path=/mnt/nas1/hf/models--Qwen--Qwen1.5-0.5B-Chat/snapshots/6114e9c18dac0042fa90925f03b046734369472f/", 
         "--model_type=qwen_2",
         '--lora_info={"test1": "/mnt/nas1/lora/qwen_lora_18000/lora"}',
         f"--start_port={port}"
         ]
    )
wait_sever_done(server_process, port)

In [ ]:
url = f"http://localhost:{port}"
json_data = {
     "prompt": "who are you",
     "generate_config": {"max_new_tokens": 32, "temperature": 0}
}

response = requests.post(url, json=json_data)
# Check each response_batch item for an error field; HTTP 200 may include per-item failures.
print(f"Output 0: {response.json()}")

In [ ]:
stop_server(server_process)

## Multiple adapters or disabled merging
With multiple entries in `--lora_info`, or with `--merge_lora false`, the service starts with the base checkpoint. None of the configured adapters is merged or selected. Requests without an adapter name use those base weights.

A request with a nonempty `generate_config.adapter_name` is rejected unless it names the single adapter already merged at startup. Naming that adapter does not switch weights; unnamed requests use it too. To serve multiple adapters concurrently, run one statically merged service per adapter and route requests to the matching service.